# Ensemble and Holdout Predictions

Train on the training split, evaluate ensembles on validation and test splits, and write holdout test predictions.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
from src.config import CFG
from src.ensemble import create_prediction_file, evaluate_predictions, rank_weighted_average
from src.modeling import MD
from src.preprocessing import get_categorical_columns, load_training_data, split_dataset

In [3]:
dataframe = load_training_data(CFG.train_path)
train_data, validation_data, test_data = split_dataset(
    dataframe,
    validation_size=CFG.validation_size,
    test_size=CFG.test_size,
    random_state=CFG.random_state,
    stratify_column=CFG.split_stratify_column,
)
cat_cols = get_categorical_columns(train_data)

print(f"Training shape: {train_data.shape}")
print(f"Validation shape: {validation_data.shape}")
print(f"Test shape: {test_data.shape}")

Training shape: (20160, 60)
Validation shape: (4320, 60)
Test shape: (4320, 60)


## Split analysis
The split keeps 70% of rows for training and reserves 15% each for validation and test. The holdout test score is therefore measured on labels that were not used during model fitting.


In [4]:
md = MD(CFG.color, train_data, cat_cols, CFG.early_stop, CFG.penalizer, CFG.n_splits, CFG.random_state)
train_data = md.create_targets()

C:\Users\hp\.conda\envs\myenv\Lib\site-packages\lifelines\utils\__init__.py:1100: ConvergenceWarning:

Column(s) ['gvhd_proph_FK+- others(not MMF,MTX)'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.




C:\Users\hp\.conda\envs\myenv\Lib\site-packages\lifelines\utils\__init__.py:1100: ConvergenceWarning:

Column(s) ['gvhd_proph_FK+- others(not MMF,MTX)'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.




C:\Users\hp\.conda\envs\myenv\Lib\site-packages\lifelines\utils\__init__.py:1100: ConvergenceWarning:

Column(s) ['gvhd_proph_FK+- others(not MMF,MTX)'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.




C:\Users\hp\.conda\envs\myenv\Lib\site-packages\lifelines\utils\__init__.py:1100: ConvergenceWarning:

Column(s) ['gvhd_proph_FK+- others(not MMF,MTX)'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.




C:\Users\hp\.conda\envs\myenv\Lib\site-packages\lifelines\utils\__init__.py:1100: ConvergenceWarning:

Column(s) ['gvhd_proph_FK+- others(not MMF,MTX)'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.




C:\Users\hp\.conda\envs\myenv\Lib\site-packages\lifelines\utils\__init__.py:1100: ConvergenceWarning:

Column(s) ['gvhd_proph_FK+- others(not MMF,MTX)'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.




Overall Stratified C-Index Score for Cox: 0.6551


Overall Stratified C-Index Score for Kaplan-Meier: 0.9987


Overall Stratified C-Index Score for Nelson-Aalen: 0.9987


## Target creation analysis
Targets are created only for the training split. Validation and test rows keep the original survival labels for unbiased holdout evaluation.


## Train base models

In [5]:
rf1_models, rf1_oof_preds = md.train_model(CFG.rf_params, target="cox_hazard", title="RandomForest")
svr1_models, svr1_oof_preds = md.train_model(CFG.svr_params, target="cox_hazard", title="SVR")
ada1_models, ada1_oof_preds = md.train_model(CFG.ada_params, target="cox_hazard", title="AdaBoost")
rf2_models, rf2_oof_preds = md.train_model(CFG.rf_params, target="km_survival", title="RandomForest")
svr2_models, svr2_oof_preds = md.train_model(CFG.svr_params, target="km_survival", title="SVR")
ada2_models, ada2_oof_preds = md.train_model(CFG.ada_params, target="km_survival", title="AdaBoost")
rf3_models, rf3_oof_preds = md.train_model(CFG.rf_params, target="na_hazard", title="RandomForest")
svr3_models, svr3_oof_preds = md.train_model(CFG.svr_params, target="na_hazard", title="SVR")
ada3_models, ada3_oof_preds = md.train_model(CFG.ada_params, target="na_hazard", title="AdaBoost")

Overall Stratified C-Index Score for RandomForest: 0.6401


Overall Stratified C-Index Score for SVR: 0.5769


Overall Stratified C-Index Score for AdaBoost: 0.6243


Overall Stratified C-Index Score for RandomForest: 0.6483


Overall Stratified C-Index Score for SVR: 0.5993


Overall Stratified C-Index Score for AdaBoost: 0.6269


Overall Stratified C-Index Score for RandomForest: 0.6494


Overall Stratified C-Index Score for SVR: 0.5804


Overall Stratified C-Index Score for AdaBoost: 0.6233


## Base model analysis
The base models cover three survival targets and three model families. This gives the ensemble several independent ranking signals instead of relying on one target definition.


## Evaluate out-of-fold ensemble

In [6]:
oof_predictions = [
    rf1_oof_preds,
    svr1_oof_preds,
    ada1_oof_preds,
    rf2_oof_preds,
    svr2_oof_preds,
    ada2_oof_preds,
    rf3_oof_preds,
    svr3_oof_preds,
    ada3_oof_preds,
]
oof_scores = [0.5645, 0.5742, 0.5584, 0.6457, 0.6010, 0.6238, 0.6473, 0.5801, 0.6216]
ensemble_oof_preds = rank_weighted_average(oof_predictions, oof_scores)
md.targets.validate_model(ensemble_oof_preds, "Training OOF Ensemble")

Overall Stratified C-Index Score for Training OOF Ensemble: 0.6419


0.6419466888691769

## OOF ensemble analysis
The out-of-fold ensemble estimates how the weighted rank blend behaves on training folds. It is useful for tuning weights, but the validation and test splits are better indicators of generalization.


## Evaluate validation and test splits

In [7]:
base_models = [
    (rf1_models, "RandomForest"),
    (svr1_models, "SVR"),
    (ada1_models, "AdaBoost"),
    (rf2_models, "RandomForest"),
    (svr2_models, "SVR"),
    (ada2_models, "AdaBoost"),
    (rf3_models, "RandomForest"),
    (svr3_models, "SVR"),
    (ada3_models, "AdaBoost"),
]
holdout_weights = [0.55, 1.0, 1.0, 0.55, 1.0, 1.0, 0.55, 2.0, 2.0]

validation_predictions = [md.infer_model(validation_data, models, title) for models, title in base_models]
ensemble_validation_preds = rank_weighted_average(validation_predictions, holdout_weights)
evaluate_predictions(validation_data, ensemble_validation_preds, "Validation Ensemble")

test_predictions = [md.infer_model(test_data, models, title) for models, title in base_models]
ensemble_test_preds = rank_weighted_average(test_predictions, holdout_weights)
evaluate_predictions(test_data, ensemble_test_preds, "Test Ensemble")

Stratified C-Index Score for Validation Ensemble: 0.6279


Stratified C-Index Score for Test Ensemble: 0.6173


0.6172878332222206

## Holdout evaluation analysis
The validation score is slightly higher than the final test score in this run. That gap suggests the ensemble generalizes reasonably, but the weights could still be tuned against validation performance.


## Write holdout test predictions

In [8]:
prediction_file = create_prediction_file(test_data, ensemble_test_preds, CFG.holdout_prediction_path)
display(prediction_file.head())

,ID,prediction
0,8743,1956.797927
1,8513,3046.735751
2,22610,3197.129534
3,15356,1345.709845
4,19370,1432.502591


## Prediction file analysis
The output file contains IDs from the internal holdout test split, not an external competition test file. It is mainly for inspection and downstream comparison inside this project.
